# GP / NTK toy regression with proper heteroscedastic NN loss
This notebook compares:
1. Infinite-width Bayesian inference (NNGP)
2. Infinite-width gradient descent (NTK)
3. Finite neural network with a **proper heteroscedastic Gaussian NLL**.

Epistemic-weighted/"heteroscedastic" NTK losses have been removed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import jax.numpy as jnp
from jax import random, jit, grad
from jax.example_libraries import optimizers

import neural_tangents as nt
from neural_tangents import stax

In [ ]:
# --- Plot helpers ---
from IPython.display import set_matplotlib_formats
set_matplotlib_formats('pdf', 'svg')

def format_plot(x=None, y=None):
    ax = plt.gca()
    if x is not None:
        plt.xlabel(x)
    if y is not None:
        plt.ylabel(y)

def legend(labels=None, loc=None):
    if labels is not None:
        plt.legend(labels, loc=loc)
    else:
        plt.legend(loc=loc)

def finalize_plot(legend_loc=None):
    if legend_loc is not None:
        plt.legend(loc='upper left', bbox_to_anchor=legend_loc)
    plt.tight_layout()

def plot_fn(train, test, *fs):
    train_xs, train_ys = train
    test_xs, test_ys = test

    plt.plot(train_xs, train_ys, 'ro', markersize=8, label='train')
    plt.plot(test_xs, test_ys, 'k--', linewidth=2, label='$f(x)$')
    for f in fs:
        plt.plot(test_xs, f, linewidth=2)

## Loss used for infinite-width NTK training dynamics (expected MSE)

In [ ]:
def loss_mse_fn(predict_fn, ys, t, xs=None):
    # Expected MSE under GP posterior at time t
    mean, cov = predict_fn(t=t, get='ntk', x_test=xs, compute_cov=True)
    mean = jnp.reshape(mean, mean.shape[:1] + (-1,))
    var = jnp.diagonal(cov, axis1=1, axis2=2)
    ys = jnp.reshape(ys, (1, -1))
    return 0.5 * jnp.mean(ys**2 - 2*mean*ys + var + mean**2, axis=1)

## Create dataset

In [ ]:
key = random.PRNGKey(42)

train_points = 20
test_points  = 100
noise_scale  = 1e-1

target_fn = lambda x: jnp.sin(2 * jnp.pi * x) + 0.5 * x

# Training data
key, x_key, noise_key = random.split(key, 3)
train_xs = random.uniform(x_key, (train_points, 1))
train_ys = target_fn(train_xs) + noise_scale * random.normal(noise_key, (train_points, 1))
train = (train_xs, train_ys)

# Test data
test_xs = jnp.linspace(-0.5, 1.5, test_points).reshape(-1, 1)
test_ys = target_fn(test_xs)
test = (test_xs, test_ys)

plt.figure(figsize=(6,4))
plot_fn(train, test)
legend(loc='upper left')
finalize_plot()
plt.show()

## Infinite-width network and kernels (NNGP / NTK)

In [ ]:
# BNN / infinite-width setup for kernels
init_fn, apply_fn, kernel_fn = stax.serial(
    stax.Dense(64, W_std=1.5, b_std=0.05), stax.Erf(),
    stax.Dense(64, W_std=1.5, b_std=0.05), stax.Erf(),
    stax.Dense(64, W_std=1.5, b_std=0.05), stax.Erf(),
    stax.Dense(1,  W_std=1.5, b_std=0.05)
)

# Prior random draws (visual sanity check)
prior_draws = []
for _ in range(4):
    key, net_key = random.split(key)
    _, params = init_fn(net_key, test_xs.shape)
    prior_draws.append(apply_fn(params, test_xs))

plt.figure(figsize=(6,4))
plot_fn(train, test, *prior_draws)
legend(['train', '$f(x)$', 'prior draws'], loc='upper left')
finalize_plot()
plt.show()

### NNGP posterior (Bayesian inference)

In [ ]:
predict_fn = nt.predict.gradient_descent_mse_ensemble(
    kernel_fn, train_xs, train_ys, diag_reg=5e-6
)

nngp_mean, nngp_covariance = predict_fn(
    x_test=test_xs, get='nngp', compute_cov=True
)

nngp_mean = jnp.reshape(nngp_mean, (-1,))
nngp_std  = jnp.sqrt(jnp.diag(nngp_covariance))

plt.figure(figsize=(6,4))
plot_fn(train, test)
plt.plot(test_xs, nngp_mean, 'r-', linewidth=2, label='NNGP mean')
plt.fill_between(
    test_xs.reshape(-1),
    (nngp_mean - 2*nngp_std),
    (nngp_mean + 2*nngp_std),
    alpha=0.2, label='NNGP ±2σ'
)
legend(loc='upper left')
finalize_plot()
plt.show()

### Optional: periodic feature map for kernels

In [ ]:
def periodic_kernel_fn(kernel_fn, x1, x2, get='nngp', period=1.0):
    # periodic + linear feature map
    x1p = jnp.concatenate([jnp.sin(2*jnp.pi*x1/period), jnp.cos(2*jnp.pi*x1/period), x1], axis=-1)
    x2p = jnp.concatenate([jnp.sin(2*jnp.pi*x2/period), jnp.cos(2*jnp.pi*x2/period), x2], axis=-1)
    return kernel_fn(x1p, x2p, get=get)

periodic_predict_fn = nt.predict.gradient_descent_mse_ensemble(
    lambda x1, x2, get: periodic_kernel_fn(kernel_fn, x1, x2, get=get),
    train_xs, train_ys, diag_reg=5e-6
)

per_nngp_mean, per_nngp_cov = periodic_predict_fn(
    x_test=test_xs, get='nngp', compute_cov=True
)
per_nngp_mean = per_nngp_mean.reshape(-1)
per_nngp_std  = jnp.sqrt(jnp.diag(per_nngp_cov))

plt.figure(figsize=(6,4))
plot_fn(train, test)
plt.plot(test_xs, per_nngp_mean, 'b-', linewidth=2, label='Periodic NNGP mean')
plt.fill_between(
    test_xs.reshape(-1),
    per_nngp_mean - 2*per_nngp_std,
    per_nngp_mean + 2*per_nngp_std,
    alpha=0.2
)
legend(loc='upper left')
finalize_plot()
plt.show()

### NTK posterior (infinite-width GD)

In [ ]:
ntk_mean, ntk_covariance = predict_fn(
    x_test=test_xs, get='ntk', compute_cov=True
)
ntk_mean = ntk_mean.reshape(-1)
ntk_std  = jnp.sqrt(jnp.diag(ntk_covariance))

plt.figure(figsize=(6,4))
plot_fn(train, test)
plt.plot(test_xs, ntk_mean, 'g-', linewidth=2, label='NTK mean')
plt.fill_between(
    test_xs.reshape(-1),
    ntk_mean - 2*ntk_std,
    ntk_mean + 2*ntk_std,
    alpha=0.2
)
legend(loc='upper left')
finalize_plot()
plt.show()

### NTK training dynamics (MSE only)

In [ ]:
ts = jnp.arange(0, 1e3, 1e-1)
ntk_train_loss_mean = loss_mse_fn(predict_fn, train_ys, ts)
ntk_test_loss_mean  = loss_mse_fn(predict_fn, test_ys, ts, test_xs)

plt.figure(figsize=(6,4))
plt.loglog(ts, ntk_train_loss_mean, linewidth=2, label='Infinite Train (NTK)')
plt.loglog(ts, ntk_test_loss_mean,  linewidth=2, label='Infinite Test (NTK)')
format_plot('Step (t)', 'Expected MSE')
legend(loc='upper right')
finalize_plot()
plt.show()

## Finite NN with proper heteroscedastic Gaussian NLL

In [ ]:
def phi(x, K=1):
    x = jnp.atleast_2d(x)
    w = 2 * jnp.pi * x
    feats = [x] + [jnp.sin(k*w) for k in range(1, K+1)] + [jnp.cos(k*w) for k in range(1, K+1)]
    return jnp.concatenate(feats, axis=-1)   # (n, 1 + 2K)

# map inputs once
Xtr, ytr = train
Xte, yte = test
Xtr_phi, Xte_phi = phi(Xtr), phi(Xte)

In [ ]:
# Finite-width NN that outputs (mean, log-variance)
init_nn_fn, apply_nn_fn = stax.serial(
    stax.Dense(128), stax.Erf(),
    stax.Dense(128), stax.Erf(),
    stax.Dense(2)  # mean and log-variance head
)

learning_rate = 1e-3
training_steps = 10_000

opt_init, opt_update, get_params = optimizers.sgd(learning_rate)
opt_update = jit(opt_update)

In [ ]:
def het_nll_loss(params, x, y, eps=1e-6):
    out = apply_nn_fn(params, phi(x))  # (N,2)
    mu = out[..., 0]
    logvar_unconstrained = out[..., 1]

    # stable positive variance
    var = jnp.softplus(logvar_unconstrained) + eps
    nll = 0.5 * ((y.squeeze() - mu.squeeze())**2 / var + jnp.log(var))
    return jnp.mean(nll)

loss_het = jit(lambda params, x, y: het_nll_loss(params, x, y))
grad_loss_het = jit(lambda state, x, y: grad(loss_het)(get_params(state), x, y))

In [ ]:
train_losses_het = []
test_losses_het = []
snapshots_het = {}

eval_steps = [0, 256, 4096, training_steps]

key = random.PRNGKey(0)
_, params = init_nn_fn(key, (-1, Xtr_phi.shape[-1]))
opt_state = opt_init(params)

for i in range(training_steps + 1):
    if i in eval_steps:
        preds = apply_nn_fn(get_params(opt_state), Xte_phi)
        snapshots_het[i] = np.asarray(preds[:, 0])  # store mean only

    if i < training_steps:
        opt_state = opt_update(i, grad_loss_het(opt_state, *train), opt_state)
        train_losses_het.append(loss_het(get_params(opt_state), *train))
        test_losses_het.append(loss_het(get_params(opt_state), *test))

In [ ]:
# Finite NN predictions
out = apply_nn_fn(get_params(opt_state), phi(test_xs))
nn_mean = out[:, 0]
nn_std  = jnp.sqrt(jnp.softplus(out[:, 1]) + 1e-6)

plt.figure(figsize=(6,4))
plot_fn(train, test)
plt.plot(test_xs, nn_mean, 'k-', linewidth=2, label='Finite NN mean')
plt.fill_between(
    test_xs.reshape(-1),
    nn_mean - 2*nn_std,
    nn_mean + 2*nn_std,
    color='k', alpha=0.15, label='Finite NN ±2σ (aleatoric)'
)
plt.plot(test_xs, nngp_mean, linestyle='-', linewidth=2, label='NNGP')
plt.plot(test_xs, ntk_mean, linestyle='-', linewidth=2, label='NTK')
legend(loc='upper left')
finalize_plot()
plt.show()

In [ ]:
# Compare loss curves (infinite NTK vs finite heteroscedastic NN)
plt.figure(figsize=(6,4))
plt.loglog(ts, ntk_train_loss_mean, linewidth=2, label='Infinite Train (NTK)')
plt.loglog(ts, ntk_test_loss_mean,  linewidth=2, label='Infinite Test (NTK)')
plt.loglog(ts[1:], train_losses_het, 'k-', linewidth=1.8, label='Finite het NN (train)')
plt.loglog(ts[1:], test_losses_het,  'k--', linewidth=1.8, label='Finite het NN (test)')
format_plot('Step (t)', 'Loss')
legend(loc='upper right')
finalize_plot()
plt.show()